load the Bronze Delta data:

In [0]:
bronze_path = "/Volumes/smart_fraud_databricks/default/bronze_data"

accounts_bronze = spark.read.format("delta").load(
    f"{bronze_path}/accounts"
)

transactions_bronze = spark.read.format("delta").load(
    f"{bronze_path}/transactions"
)

watchlist_bronze = spark.read.format("delta").load(
    f"{bronze_path}/fraud_watchlist"
)

print("Bronze data loaded successfully")

Bronze data loaded successfully


Step 2 — Remove duplicate records

In [0]:
accounts_silver = accounts_bronze.dropDuplicates()

transactions_silver = transactions_bronze.dropDuplicates()

watchlist_silver = watchlist_bronze.dropDuplicates()

print("Duplicates removed")

Duplicates removed


Step 3 — Remove completely empty rows

In [0]:
from pyspark.sql.functions import *

accounts_silver = accounts_silver.dropna(how="all")

transactions_silver = transactions_silver.dropna(how="all")

watchlist_silver = watchlist_silver.dropna(how="all")

In [0]:
spark.sql("SHOW CATALOGS").show()

+--------------------+
|             catalog|
+--------------------+
|             samples|
|smart_fraud_datab...|
|              system|
+--------------------+



In [0]:
spark.sql("SHOW VOLUMES IN smart_fraud_databricks.default").show()

+--------+-----------+
|database|volume_name|
+--------+-----------+
| default|   raw_data|
+--------+-----------+



In [0]:
bronze_path = "/Volumes/smart_fraud_databricks/default/raw_data/bronze"
silver_path = "/Volumes/smart_fraud_databricks/default/raw_data/silver"

In [0]:
display(dbutils.fs.ls("/Volumes/smart_fraud_databricks/default/raw_data"))

path,name,size,modificationTime
dbfs:/Volumes/smart_fraud_databricks/default/raw_data/accounts.csv,accounts.csv,21976,1786201671000
dbfs:/Volumes/smart_fraud_databricks/default/raw_data/bronze/,bronze/,0,1786203216000
dbfs:/Volumes/smart_fraud_databricks/default/raw_data/fraud_watchlist.csv,fraud_watchlist.csv,1576,1786201671000
dbfs:/Volumes/smart_fraud_databricks/default/raw_data/transactions.csv,transactions.csv,963914,1786201673000


In [0]:
raw_path = "/Volumes/smart_fraud_databricks/default/raw_data"

accounts_bronze = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{raw_path}/accounts.csv")

transactions_bronze = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{raw_path}/transactions.csv")

watchlist_bronze = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{raw_path}/fraud_watchlist.csv")

In [0]:
accounts_bronze.write.format("delta").mode("overwrite").save(
    f"{bronze_path}/accounts"
)

transactions_bronze.write.format("delta").mode("overwrite").save(
    f"{bronze_path}/transactions"
)

watchlist_bronze.write.format("delta").mode("overwrite").save(
    f"{bronze_path}/fraud_watchlist"
)

print("Bronze layer created successfully!")

Bronze layer created successfully!


In [0]:
accounts_bronze = spark.read.format("delta").load(
    f"{bronze_path}/accounts"
)

transactions_bronze = spark.read.format("delta").load(
    f"{bronze_path}/transactions"
)

watchlist_bronze = spark.read.format("delta").load(
    f"{bronze_path}/fraud_watchlist"
)

print("Bronze data loaded successfully!")

Bronze data loaded successfully!


Step 4 — Clean the Bronze data

In [0]:
from pyspark.sql.functions import col, count, when

accounts_silver = accounts_bronze.dropDuplicates().dropna(how="all")
transactions_silver = transactions_bronze.dropDuplicates().dropna(how="all")
watchlist_silver = watchlist_bronze.dropDuplicates().dropna(how="all")

print("Silver cleaning completed")

Silver cleaning completed


Step 5 — Check NULL values

In [0]:
def null_report(df):
    return df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df.columns
    ])

print("Accounts NULL report")
display(null_report(accounts_silver))

print("Transactions NULL report")
display(null_report(transactions_silver))

print("Watchlist NULL report")
display(null_report(watchlist_silver))

Accounts NULL report


account_id,customer_name,account_type,credit_limit,branch
1,1,0,0,1


Transactions NULL report


txn_id,account_id,txn_date,amount,merchant
0,1,1,1,0


Watchlist NULL report


account_id,fraud_type,flagged_date
0,1,0


Step 6 — Save Silver Delta

In [0]:
silver_path = "/Volumes/smart_fraud_databricks/default/raw_data/silver"

In [0]:
accounts_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{silver_path}/accounts")

transactions_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{silver_path}/transactions")

watchlist_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{silver_path}/fraud_watchlist")

print("Silver Delta layer created successfully!")

Silver Delta layer created successfully!


Step 7 — Verify

In [0]:
print("Accounts:", spark.read.format("delta")
      .load(f"{silver_path}/accounts").count())

print("Transactions:", spark.read.format("delta")
      .load(f"{silver_path}/transactions").count())

print("Watchlist:", spark.read.format("delta")
      .load(f"{silver_path}/fraud_watchlist").count())

Accounts: 505
Transactions: 20010
Watchlist: 45
